In [ ]:
!pip install tensorflowjs scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.1/89.1 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 1.7 MB/s eta 0:00:00
  Attempting uninstall: packaging
    Found existing installation: packaging 24.2
    Uninstalling packaging-24.2:
      Successfully uninstalled packaging-24.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-bigquery 3.33.0 requires packaging>=24.2.0, but you have packaging 23.2 which is incompatible.
db-dtypes 1.4.3 requires packaging>=24.2.0, but you have packaging 23.2 which is incompatible.


In [ ]:
\import pandas as pd
import numpy as np
import joblib
import tensorflow as tf
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout
import tensorflowjs as tfjs

In [ ]:
df = pd.read_csv("/content/Dataset_Stunting.csv").dropna()

In [ ]:
# === 2. ENCODING LABEL KELAS ===
label_encoder = LabelEncoder()
df["status_gizi_who_encoded"] = label_encoder.fit_transform(df["status_gizi_who"])
y = df["status_gizi_who_encoded"]

In [ ]:
# === 3. ENCODING FITUR KATEGORIK ===
for col in df.select_dtypes(include="object").columns:
    if col not in ["status_gizi_who"]:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col])

In [ ]:
# === 4. FITUR YANG DIGUNAKAN (HANYA 4) ===
fitur = ["umur", "berat_badan", "tinggi_badan", "jenis_kelamin"]
X = df[fitur]

In [ ]:
# === 5. NORMALISASI ===
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
# === 6. CLASS WEIGHT ===
class_weights = compute_class_weight(class_weight="balanced", classes=np.unique(y), y=y)
class_weights_dict = dict(enumerate(class_weights))
print("\n📊 Distribusi Kelas dan Bobot:", class_weights_dict)


📊 Distribusi Kelas dan Bobot: {0: np.float64(0.4599163010672392), 1: np.float64(1.831629491945477), 2: np.float64(3.5748836084406554)}


In [ ]:
# === 7. SPLIT DATA (TEST SET) ===
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, stratify=y, random_state=42
)

In [ ]:
# === 8. STRATIFIED K-FOLD VALIDATION (tambahan evaluasi) ===
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
f1_scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    print(f"\n🔁 Fold {fold+1}")
    model = Sequential([
        Input(shape=(4,)),  # TETAP 4 FITUR INPUT
        Dense(128, activation='relu'),
        Dropout(0.3),
        Dense(64, activation='relu'),
        Dropout(0.2),
        Dense(3, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    model.fit(
        X_train[train_idx], y_train.iloc[train_idx],
        validation_data=(X_train[val_idx], y_train.iloc[val_idx]),
        epochs=30, batch_size=32,
        class_weight=class_weights_dict,
        verbose=0
    )

    y_val_pred = np.argmax(model.predict(X_train[val_idx]), axis=1)
    f1 = f1_score(y_train.iloc[val_idx], y_val_pred, average='weighted')
    f1_scores.append(f1)
    print(f"✅ Weighted F1-score Fold {fold+1}: {f1:.3f}")

print(f"\n📌 Rata-rata Weighted F1-Score (CV): {np.mean(f1_scores):.3f}")



🔁 Fold 1
296/296 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
✅ Weighted F1-score Fold 1: 0.952

🔁 Fold 2
296/296 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step
✅ Weighted F1-score Fold 2: 0.957

🔁 Fold 3
296/296 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
✅ Weighted F1-score Fold 3: 0.945

🔁 Fold 4
296/296 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
✅ Weighted F1-score Fold 4: 0.955

🔁 Fold 5
296/296 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step
✅ Weighted F1-score Fold 5: 0.954

📌 Rata-rata Weighted F1-Score (CV): 0.952


In [ ]:
# === 9. TRAIN FINAL MODEL (UNTUK SIMPAN) ===
model = Sequential([
    Input(shape=(4,)),  # SESUAI UNTUK WEBSITE
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(3, activation='softmax')
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(X_train, y_train, epochs=40, batch_size=32, validation_split=0.2, class_weight=class_weights_dict)


Epoch 1/40
1183/1183 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.8045 - loss: 0.6140 - val_accuracy: 0.8924 - val_loss: 0.2643
Epoch 2/40
1183/1183 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.8901 - loss: 0.3543 - val_accuracy: 0.9178 - val_loss: 0.1999
Epoch 3/40
1183/1183 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.9022 - loss: 0.3041 - val_accuracy: 0.9162 - val_loss: 0.1996
Epoch 4/40
1183/1183 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.9119 - loss: 0.2745 - val_accuracy: 0.9239 - val_loss: 0.1805
Epoch 5/40
1183/1183 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9168 - loss: 0.2645 - val_accuracy: 0.9253 - val_loss: 0.1714
Epoch 6/40
1183/1183 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.9164 - loss: 0.2412 - val_accuracy: 0.9208 - val_loss: 0.1748
Epoch 7/40
1183/1183 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.9162 - loss: 0.2409 - val_accuracy: 0.9283 - val_loss: 0.1605
Epoch 8/40
1183/1183 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.9229 - loss: 0.2348 - 

In [ ]:
# === 10. EVALUASI AKHIR DI DATA UJI ===
loss, accuracy = model.evaluate(X_test, y_test)
print(f"\n✅ Akurasi pada Data Uji: {accuracy:.3f}")

y_pred_probs = model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)

print("\n📌 Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\n📋 Classification Report:\n", classification_report(y_test, y_pred, target_names=label_encoder.classes_))
print(f"🎯 F1-score (weighted): {f1_score(y_test, y_pred, average='weighted'):.3f}")


370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9533 - loss: 0.0974

✅ Akurasi pada Data Uji: 0.953
370/370 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step

📌 Confusion Matrix:
 [[8168    2  400]
 [   0 2018  134]
 [   0   21 1082]]

📋 Classification Report:
                   precision    recall  f1-score   support

          normal       1.00      0.95      0.98      8570
severely stunted       0.99      0.94      0.96      2152
         stunted       0.67      0.98      0.80      1103

        accuracy                           0.95     11825
       macro avg       0.89      0.96      0.91     11825
    weighted avg       0.97      0.95      0.96     11825

🎯 F1-score (weighted): 0.957


In [ ]:
# === 11. SIMPAN MODEL & SCALER ===
model.save("model_stunting_nn.h5")
joblib.dump(scaler, "scaler_stunting.pkl")
joblib.dump(label_encoder, "label_encoder_stunting.pkl")
joblib.dump(class_weights_dict, "class_weights_stunting.pkl")

tfjs.converters.save_keras_model(model, "tfjs_model")
print("\n✅ Model berhasil dikonversi ke TensorFlow.js di folder 'tfjs_model/'")


failed to lookup keras version from the file,
    this is likely a weight only file

✅ Model berhasil dikonversi ke TensorFlow.js di folder 'tfjs_model/'
